# v2/backtest — No-Refill variant

| | |
|---|---|
| **Version** | v2_no_refill |
| **Key change** | Same signals as baseline but **no capital refills** — if capital < 1-lot cost, that trade is skipped |
| **Data source** | 2024 real 1-min NSE option CSVs |
| **Lookahead** | None — all global signals use D-1 daily close |
| **Signals CSV** | `v2/v2_reliable_signals.csv` (top-10 bearish combos, original) |

Reuses `trade_cache_v2_backtest.pkl` — signal days are identical to the baseline.
Re-run Cell 4–5 freely to test different SL/TP without rebuilding the cache.


In [21]:
# ╔══════════════════════════════════╗
# ║  EDIT THESE — re-run Cell 4+5   ║
# ╚══════════════════════════════════╝
VERSION    = 'v2_no_refill'
KEY_CHANGE = 'No capital refills — skip trade if capital < 1-lot cost'
DATA_SRC   = 'real_2024_options'

SL_PCT    = 0.15    # stop loss  15%
TP_PCT    = 0.40    # target     40%
BASE_LOTS = 2
MAX_LOTS  = 10
DTE0_LOTS = 5
LOT_SIZE  = 75
BEAR_N    = 10      # top-N bearish combos
BASE_RATE = 54.5
ENTRY_TIME      = '09:25'
EXIT_TIME       = '11:15'
INITIAL_CAPITAL = 10000.0   # fixed starting capital for all versions (Rs)
NO_REFILL       = True     # skip trade instead of topping up capital

print(f"Breakeven win rate: {SL_PCT/(SL_PCT+TP_PCT)*100:.1f}%")
print(f"No-refill mode: {NO_REFILL}")


Breakeven win rate: 27.3%
No-refill mode: True


In [22]:
import sys, os
from pathlib import Path

HERE    = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
V4_DIR  = HERE.parent if HERE.name.startswith('v4') else HERE.parent.parent / 'v4'
sys.path.insert(0, str(V4_DIR))

from _engine import (simulate_trades, compute_metrics, save_results,
                     sig_map, combo_fires, load_reliable_signals, print_summary)

import pandas as pd
import numpy  as np
import yfinance as yf
import pickle, warnings
from datetime import date, timedelta, time as dtime
from pathlib import Path
warnings.filterwarnings('ignore')


In [23]:
# ── Paths ──────────────────────────────────────────────────────────────────────
GAP_TRADING     = HERE.parent.parent
DATA_2024       = GAP_TRADING / 'v2' / 'backtesting_2024_options' / '2024'
SIGNALS_CSV     = GAP_TRADING / 'v2' / 'v2_reliable_signals.csv'
NIFTY_SPOT_DIR  = DATA_2024 / '2024Nifty'
EXPIRY_CSV      = DATA_2024 / 'expiry.csv'
# Reuse the baseline cache — signal days are identical (no refill only affects simulation)
CACHE_FILE = HERE / 'results' / 'trade_cache_v2_backtest.pkl'

# ── NSE holidays and event days (2024) ────────────────────────────────────────
NSE_HOLIDAYS = {
    date(2024,  1, 22), date(2024,  3, 25), date(2024,  3, 29),
    date(2024,  4, 14), date(2024,  4, 17), date(2024,  5,  1),
    date(2024,  6, 17), date(2024,  7, 17), date(2024,  8, 15),
    date(2024, 10,  2), date(2024, 11, 15), date(2024, 12, 25),
}
EVENT_DAYS = {
    date(2024,  2,  1),
    date(2024,  2,  8),
    date(2024,  4,  5),
    date(2024,  6,  7),
    date(2024,  8,  8),
    date(2024, 10,  9),
    date(2024, 12,  6),
}

# ── Expiry helpers ─────────────────────────────────────────────────────────────
expiry_df = pd.read_csv(EXPIRY_CSV)
expiry_df.columns = [c.strip() for c in expiry_df.columns]
_ecol = [c for c in expiry_df.columns if 'expiry' in c.lower() or 'date' in c.lower()][0]
expiry_dates = sorted(
    pd.to_datetime(expiry_df[_ecol].str.strip(), format='%d%b%y').dt.date.tolist()
)

def nearest_expiry(d):
    for e in expiry_dates:
        if e >= d:
            return e
    return expiry_dates[-1]

def expiry_folder(exp):
    return '2024' + exp.strftime('%b').capitalize()

# ── Global market data ─────────────────────────────────────────────────────────
print("Fetching global market data ...", end=' ', flush=True)
START = '2023-12-15'
END   = '2025-01-10'
TICKERS = {'SP500': '^GSPC', 'SGX': 'NKD=F', 'DAX': '^GDAXI',
           'VIX': '^VIX', 'NIFTY': '^NSEI'}

raw = {}
for name, ticker in TICKERS.items():
    try:
        df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df.index.tz is None:
            df.index = df.index.tz_localize('UTC')
        raw[name] = df[['Close']].rename(columns={'Close': name})
    except Exception as e:
        print(f"  [warn] {name}: {e}")
        raw[name] = pd.DataFrame()

gdf = pd.concat([v for v in raw.values() if not v.empty], axis=1)
gdf.index = gdf.index.date
gdf = gdf.ffill()
print(f"done. {len(gdf)} rows.")

# ── Load signal combos ────────────────────────────────────────────────────────
bear_combos = load_reliable_signals(SIGNALS_CSV, bear_n=BEAR_N, base_rate=BASE_RATE)
print(f"Loaded {len(bear_combos)} bearish combos.")

# ── NIFTY spot 1-min data ─────────────────────────────────────────────────────
spot_cache = {}
for f in sorted(NIFTY_SPOT_DIR.glob('Nifty-2024*.csv')):
    try:
        s = pd.read_csv(f)
        s.columns = [c.strip().lower() for c in s.columns]
        s['datetime'] = pd.to_datetime(s['datetime'])
        s['date'] = s['datetime'].dt.date
        s['time'] = s['datetime'].dt.time
        for d, grp in s.groupby('date'):
            spot_cache[d] = grp.reset_index(drop=True)
    except Exception as e:
        print(f"  [warn] spot {f.name}: {e}")
print(f"Spot data loaded for {len(spot_cache)} days.")

# ── Load signal_days from baseline cache ──────────────────────────────────────
if CACHE_FILE.exists():
    with open(CACHE_FILE, 'rb') as f:
        signal_days = pickle.load(f)
    valid = sum(1 for v in signal_days.values() if v is not None)
    print(f"Loaded baseline cache: {len(signal_days)} dates, {valid} with signal.")
else:
    raise FileNotFoundError(
        f"Baseline cache not found at {CACHE_FILE}\n"
        "Run backtest.ipynb first to build the cache."
    )


Fetching global market data ... done. 277 rows.
Loaded 10 bearish combos.
Spot data loaded for 245 days.
Loaded baseline cache: 189 dates, 57 with signal.


In [24]:
# ── Parameters dict passed to engine ──────────────────────────────────────────
params = dict(
    SL_PCT          = SL_PCT,
    TP_PCT          = TP_PCT,
    BASE_LOTS       = BASE_LOTS,
    MAX_LOTS        = MAX_LOTS,
    DTE0_LOTS       = DTE0_LOTS,
    LOT_SIZE        = LOT_SIZE,
    BEAR_N          = BEAR_N,
    ENTRY_TIME      = ENTRY_TIME,
    EXIT_TIME       = EXIT_TIME,
    INITIAL_CAPITAL = INITIAL_CAPITAL,
    NO_REFILL       = NO_REFILL,      # skip trade if capital insufficient
)

df_log = simulate_trades(signal_days, params)
print(f"Simulation complete: {len(df_log)} trades executed.")
df_log


Simulation complete: 56 trades executed.


,trade_num,date,strike,dte,entry,lots,sl_price,tp_price,exit_price,exit_reason,exit_time,pnl_pts,pnl_rs,charges_rs,capital_before,capital_after,drawdown_pct,combo,refill_rs
0,1,2024-01-04,21550,0,19.95,5,16.9575,27.93,16.9575,Stop Loss,09:28:00,-2.9925,-1182.2584,60.0709,10000.0000,8817.7416,11.8226,Gap Up + Prev India DOWN + SGX UP,0.0
1,2,2024-01-05,21650,6,82.10,1,69.7850,114.94,94.3000,11:15 exit,11:15:00,12.2000,854.9053,60.0947,8817.7416,9672.6469,3.2735,Gap Up + SGX UP + DAX UP,0.0
2,3,2024-01-09,21600,2,55.40,2,47.0900,77.56,77.5600,Target Hit,09:44:00,22.1600,3256.7829,67.2171,9672.6469,12929.4298,0.0000,Gap Up + Prev India DOWN + US UP + SGX UP,0.0
3,4,2024-01-11,21650,0,18.95,5,16.1075,26.53,16.1075,Stop Loss,09:26:00,-2.8425,-1125.3633,59.4258,12929.4298,11804.0665,8.7039,Gap Up + SGX UP + DAX UP + VIX Falling,0.0
4,5,2024-01-12,21700,6,105.05,1,89.2925,147.07,89.2925,Stop Loss,10:04:00,-15.7575,-1242.5673,60.7548,11804.0665,10561.4992,18.3143,Gap Up + SGX UP + VIX Falling,0.0
5,6,2024-01-19,21600,6,124.50,1,105.8250,174.30,105.8250,Stop Loss,10:31:00,-18.6750,-1463.8894,63.2644,10561.4992,9097.6098,29.6364,Gap Up + Prev India DOWN + US UP + SGX UP,0.0
6,7,2024-01-23,21650,2,61.70,1,52.4450,86.38,86.3800,Target Hit,10:00:00,24.6800,1792.6533,58.3467,9097.6098,10890.2631,15.7715,Gap Up + SGX UP + DAX UP + VIX Falling,0.0
7,8,2024-02-02,21900,6,128.70,1,109.3950,180.18,109.3950,Stop Loss,10:10:00,-19.3050,-1511.6814,63.8064,10890.2631,9378.5817,27.4633,Gap Up + Prev India DOWN + US UP + SGX UP,0.0
8,9,2024-02-07,21950,1,67.45,1,57.3325,94.43,94.4300,Target Hit,09:45:00,26.9800,1964.1145,59.3855,9378.5817,11342.6962,12.2723,Gap Up + DAX UP + VIX Falling,0.0
9,10,2024-02-13,21600,2,117.60,1,99.9600,164.64,164.6400,Target Hit,09:31:00,47.0400,3459.5545,68.4455,11342.6962,14802.2507,0.0000,Gap Up + Prev India DOWN + SGX UP + DAX UP,0.0


In [25]:
metrics = compute_metrics(df_log, params)
print_summary(df_log, metrics, params)

version_meta = dict(
    version     = VERSION,
    key_change  = KEY_CHANGE,
    data_source = DATA_SRC,
)
out = save_results(metrics, params, version_meta, HERE / 'results')
print(f"\nRun ID: {out.stem}")


  BACKTEST RESULTS  ·  2024-01-04 to 2024-12-05
  Trades          : 56
  TP hit          : 33.9%  (19 trades)
  SL hit          : 62.5%  (35 trades)
  Time exit       : 3.6%  (2 trades)
  Profitable      : 37.5%
  Avg win (Rs)    :      3,828
  Avg loss (Rs)   :     -1,979
  Total P&L (Rs)  :     11,123
  Net return      : +111.2%
  XIRR            : +125.3%
  Max drawdown    : 41.0%
  Starting cap    : Rs     10,000
  Ending cap      : Rs     21,123
  Breakeven WR    : 27.3%  (SL=15% / TP=40%)

  Month     Trades   Wins    Win%      P&L (Rs)
  --------  ------  -----  ------  ------------
  2024-01        7      3   42.9%           890
  2024-02        6      3   50.0%         4,828
  2024-03        5      1   20.0%        -2,690
  2024-04        6      2   33.3%          -100
  2024-05        3      2   66.7%         5,985
  2024-06        4      1   25.0%        -1,962
  2024-07        4      1   25.0%        -1,427
  2024-08        7      3   42.9%          -348
  2024-09        3 